# only 2 states lol

## not that much static stuff here

In [ ]:
import numpy as np
from tqdm import tqdm

log_sum = np.logaddexp.reduce  # not sure where else to place this

In [ ]:
NUM_STATES = 2  # 0 = B, 1 = P
EMIT_CHARS = "nxyz"
CHARS_TO_NUM = {c: i for i, c in enumerate(EMIT_CHARS)}

## reading data

In [ ]:
with open("data/gene_marks.fasta") as read:
    read.readline()  # first line is useless :OMEGALUL:
    gene_marks = np.array([CHARS_TO_NUM[c.strip()] for c in read])

like with promoter marks it's basically the same thing what

In [ ]:
with open("data/promoter_marks.fasta") as read:
    read.readline()
    promoter_marks = np.array([CHARS_TO_NUM[c.strip()] for c in read])

## tuning

idk, these initial states seem somewhat sensible

In [ ]:
TRANSITION = np.array([[0.9, 0.1], [0.25, 0.75]])
E_PROB = np.array([[0.4, 0.3, 0.3, 0.1], [0.1, 0.3, 0.3, 0.3]])

NOW we can actually train the darn thing

In [6]:
no_prior = np.log(1 / NUM_STATES)

ep = np.log(E_PROB)
prob = np.log(TRANSITION)

final_probs = []
for marks in [gene_marks, promoter_marks]:
    fwd = np.empty((len(marks), NUM_STATES))
    bkwd = np.empty((len(marks), NUM_STATES))
    sstar = np.empty((len(marks) - 1, NUM_STATES, NUM_STATES))

    bkwd[-1, :] = np.log(1)  # this case never changes
    for _ in tqdm(range(50)):
        # idek what these operations even mean anymore dog
        fwd[0] = no_prior + ep[:, marks[0]]
        for i in range(1, len(marks)):
            fwd[i] = log_sum(fwd[i - 1][:, None] + prob) + ep[:, marks[i]]
        
        for i in range(len(marks) - 2, -1, -1):
            # ??? no clue why this one needs an explicit axis argument but wtv
            bkwd[i] = log_sum(bkwd[i + 1] + prob + ep[:, marks[i + 1]], axis=1)
        
        sink = log_sum(fwd[-1])
        for i in range(len(marks) - 1):
            weight_mat = prob + ep[:, marks[i + 1]]
            sstar[i] = fwd[i][:, None] + weight_mat + bkwd[i + 1] - sink

        new_prob = log_sum(sstar)  # LOL WHAT
        new_prob -= log_sum(new_prob, axis=1)[:, None]

        star = fwd + bkwd - sink
        new_ep = np.empty_like(ep)
        for e in range(len(EMIT_CHARS)):
            new_ep[:, e] = log_sum(star[marks == e])
        new_ep -= log_sum(new_ep, axis=1)[:, None]

        prob, ep = new_prob, new_ep
    
    final_probs.append(fwd + bkwd)  # this is actually all we need

100%|██████████| 50/50 [07:53<00:00,  9.47s/it]


In [7]:
thresholds = [50000, 5000]
output = []
for raw, thresh in zip(final_probs, thresholds):
    p_prob = raw[:, 1] - np.logaddexp(raw[:, 0], raw[:, 1])
    positions = np.argsort(p_prob)[::-1]
    output.append(positions[:thresh])

In [8]:
prefixes = ["G", "P"]
with open("predictions.csv", "w") as read:
    for pos, pref in zip(output, prefixes):
        out = "\n".join(f"{pref}\t{p + 1}" for p in pos)
        print(out, file=read)